In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [3]:
!pip -q install duckdb

In [4]:
import duckdb

con = duckdb.connect()

print("DuckDB Ready!")

DuckDB Ready!


In [5]:
from huggingface_hub import hf_hub_download

sample = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print(sample)

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [6]:
con.sql(f"""
SELECT *
FROM read_parquet('{sample}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [7]:
con.sql(f"""
CREATE OR REPLACE TABLE daily AS
SELECT *
FROM read_parquet('{sample}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
con.sql("""
SELECT COUNT(*) AS total_rows
FROM daily
""").df()

,total_rows
0,11694072


In [9]:
con.sql("""
DESCRIBE daily
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Signal Check 1: CTR vs Average Position

Reason:
This signal checks whether pages with better average search positions generally receive a higher click-through rate (CTR). This is one of the real FlyRank signals used behind CTR optimization.

Verdict: **CONFIRMED**

Observation:
Pages ranking in the Top 3 positions achieved the highest CTR (2.78%). CTR declined as average position became worse, reaching 0.28% for pages ranked beyond position 20. This confirms that search position is strongly associated with click-through rate and supports using this signal in a baseline rule.

In [10]:
con.sql("""
SELECT
CASE
    WHEN gsc_avg_position <= 3 THEN 'Top 3'
    WHEN gsc_avg_position <= 10 THEN 'Top 10'
    WHEN gsc_avg_position <= 20 THEN 'Top 20'
    ELSE '20+'
END AS position_bucket,

COUNT(*) AS n,

ROUND(
100.0 * SUM(gsc_clicks) /
NULLIF(SUM(gsc_impressions),0),
2
) AS ctr

FROM daily

WHERE
gsc_data_available IS TRUE
AND gsc_impressions > 0

GROUP BY position_bucket

ORDER BY
CASE position_bucket
WHEN 'Top 3' THEN 1
WHEN 'Top 10' THEN 2
WHEN 'Top 20' THEN 3
ELSE 4
END
""").df()

,position_bucket,n,ctr
0,Top 3,440501,2.78
1,Top 10,1530594,0.42
2,Top 20,633342,0.42
3,20+,1274500,0.28


### Signal Check 2: Search Volume (Impressions)

Reason:
This signal checks whether pages with higher search impressions also receive more clicks. Search volume is commonly used to identify high-impact optimization opportunities.

Verdict: **CONFIRMED**

Observation:
Pages with 1000 or more impressions generated the highest number of clicks, while pages with fewer impressions generated progressively fewer clicks. This indicates that higher search visibility generally creates greater optimization opportunities, making impressions a useful signal for prioritizing content.

In [11]:
con.sql("""
SELECT
CASE
    WHEN gsc_impressions >= 1000 THEN '1000+'
    WHEN gsc_impressions >= 100 THEN '100-999'
    WHEN gsc_impressions >= 10 THEN '10-99'
    ELSE '0-9'
END AS impression_bucket,

COUNT(*) AS n,

SUM(gsc_clicks) AS total_clicks

FROM daily

WHERE gsc_data_available IS TRUE

GROUP BY impression_bucket

ORDER BY
CASE impression_bucket
WHEN '1000+' THEN 1
WHEN '100-999' THEN 2
WHEN '10-99' THEN 3
ELSE 4
END
""").df()

,impression_bucket,n,total_clicks
0,1000+,25519,519661.0
1,100-999,408641,417105.0
2,10-99,1526363,233064.0
3,0-9,1918414,39287.0


## 2. Baseline Rule

Rule Description:

This baseline rule prioritizes content with high search visibility but relatively low click performance. Pages with high impressions and low CTR are assigned a higher priority score because they represent potential optimization opportunities.

Reason Code:
LOW_CTR_HIGH_IMPRESSIONS

Action Label:
Optimize Title and Meta Description

In [12]:
baseline = con.sql("""
SELECT
report_date,
client_hash_id,
content_hash_id,

gsc_impressions,
gsc_clicks,

ROUND(
100.0 * gsc_clicks /
NULLIF(gsc_impressions,0),
2
) AS ctr,

CASE
WHEN gsc_impressions >= 1000
AND (
100.0 * gsc_clicks /
NULLIF(gsc_impressions,0)
) < 1
THEN 100

WHEN gsc_impressions >= 100
AND (
100.0 * gsc_clicks /
NULLIF(gsc_impressions,0)
) < 1
THEN 75

WHEN gsc_impressions >= 10
AND (
100.0 * gsc_clicks /
NULLIF(gsc_impressions,0)
) < 1
THEN 50

ELSE 10
END AS score,

'LOW_CTR_HIGH_IMPRESSIONS' AS reason_code,

'Optimize Title and Meta Description' AS action_label

FROM daily

WHERE
gsc_data_available IS TRUE
AND gsc_impressions > 0

ORDER BY score DESC,
gsc_impressions DESC

LIMIT 100
""").df()

baseline.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action_label
0,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,0.36,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
1,2026-06-29,client_e547b89c05043229,content_eadb33b5df496f4a,49373,173,0.35,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
2,2026-06-30,client_e547b89c05043229,content_eadb33b5df496f4a,48990,189,0.39,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
3,2026-06-26,client_e547b89c05043229,content_545bb6cc7081ded3,48953,114,0.23,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
4,2026-06-12,client_e547b89c05043229,content_963de14b1f58978f,46220,81,0.18,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
5,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,45890,329,0.72,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
6,2026-06-25,client_e547b89c05043229,content_545bb6cc7081ded3,42474,107,0.25,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
7,2026-06-27,client_e547b89c05043229,content_545bb6cc7081ded3,41051,129,0.31,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
8,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,35560,313,0.88,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
9,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,30645,50,0.16,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description


In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully!")

CSV written successfully!


## 3. Top 10 Review

The following table contains the top 10 pages prioritized by the baseline rule. Each recommendation includes the suggested action, why the page was selected, and what could make the recommendation incorrect.

In [14]:
baseline.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action_label
0,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,0.36,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
1,2026-06-29,client_e547b89c05043229,content_eadb33b5df496f4a,49373,173,0.35,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
2,2026-06-30,client_e547b89c05043229,content_eadb33b5df496f4a,48990,189,0.39,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
3,2026-06-26,client_e547b89c05043229,content_545bb6cc7081ded3,48953,114,0.23,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
4,2026-06-12,client_e547b89c05043229,content_963de14b1f58978f,46220,81,0.18,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
5,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,45890,329,0.72,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
6,2026-06-25,client_e547b89c05043229,content_545bb6cc7081ded3,42474,107,0.25,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
7,2026-06-27,client_e547b89c05043229,content_545bb6cc7081ded3,41051,129,0.31,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
8,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,35560,313,0.88,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description
9,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,30645,50,0.16,100,LOW_CTR_HIGH_IMPRESSIONS,Optimize Title and Meta Description


| Rank | Action | Why it's here | What would make it wrong |
|------|--------|---------------|--------------------------|
| 1 | Optimize Title and Meta Description | High impressions with low CTR produced the highest priority score. | Seasonal search behavior or misleading search intent may reduce CTR regardless of title changes. |
| 2 | Optimize Title and Meta Description | High search visibility but relatively low click performance. | CTR may already be limited by strong competitors or SERP features. |
| 3 | Optimize Title and Meta Description | High impressions indicate strong visibility with optimization potential. | Search intent may not match the page content. |
| 4 | Optimize Title and Meta Description | Large impression volume combined with low CTR. | Temporary ranking fluctuations may affect performance. |
| 5 | Optimize Title and Meta Description | Page meets the baseline rule for optimization. | Data may represent only a short reporting period. |
| 6 | Optimize Title and Meta Description | High exposure but fewer clicks than expected. | User behavior may be influenced by external events. |
| 7 | Optimize Title and Meta Description | Prioritized because of high impressions and low CTR. | CTR may improve naturally without intervention. |
| 8 | Optimize Title and Meta Description | Selected based on the scoring rule. | Missing or incomplete GSC data could affect the recommendation. |
| 9 | Optimize Title and Meta Description | Low CTR despite strong visibility. | SERP features may reduce clicks even for well-optimized pages. |
|10 | Optimize Title and Meta Description | Ranked highly by the baseline scoring rule. | Future performance may differ from the current observation period. |

## 4. Weak Picks

Potential weaknesses of this baseline rule:

- The rule relies mainly on impressions and CTR and does not consider search intent.
- Seasonal traffic patterns may influence page performance.
- Some pages may have incomplete Google Search Console or GA4 data.
- The rule does not account for competitor activity or SERP features.
- Future performance may differ from the observed reporting window, so the rule should be treated as decision support rather than a final recommendation.

## 5. Self Check

-  Two signal checks completed with bucket tables and verdicts.
-  One baseline rule created with a score, reason code, and action label.
-  Ranked queue generated and exported as `work/outputs/baseline_action_score.csv`.
-  Top 10 recommendations reviewed with explanations and possible limitations.
-  No future-window or label-derived inputs were used.
-  Notebook runs successfully from start to finish.